In [1]:
from blazingsql import BlazingContext
import cudf

bc_cudf = BlazingContext()

BlazingContext ready


In [2]:
%%time
# Load CSVs into GPU DataFrames with cudf (GDF)
netflow_gdf_cudf = cudf.read_csv('./nf-chunk2.csv')

CPU times: user 4.61 s, sys: 1.52 s, total: 6.14 s
Wall time: 6.15 s


In [3]:
%%time
# Create BlazingSQL table from GDF
bc_cudf.create_table('netflow', netflow_gdf_cudf)

CPU times: user 2.1 ms, sys: 529 µs, total: 2.63 ms
Wall time: 1.7 ms


In [4]:
df_cudf = cudf.DataFrame(netflow_gdf_cudf)

In [5]:
df_cudf

,TimeSeconds,parsedDate,dateTimeStr,ipLayerProtocol,ipLayerProtocolCode,firstSeenSrcIp,firstSeenDestIp,firstSeenSrcPort,firstSeenDestPort,moreFragments,contFragments,durationSeconds,firstSeenSrcPayloadBytes,firstSeenDestPayloadBytes,firstSeenSrcTotalBytes,firstSeenDestTotalBytes,firstSeenSrcPacketCount,firstSeenDestPacketCount,recordForceOut
0,1.364948e+09,2013-04-03 00:11:50,2.013040e+13,6,TCP,10.38.37.13,172.20.0.3,42559,25,0,0,10,36,125,422,403,7,5,0
1,1.364948e+09,2013-04-03 00:11:53,2.013040e+13,6,TCP,10.13.77.49,172.30.0.4,42566,25,0,0,9,0,0,186,0,3,0,0
2,1.364948e+09,2013-04-03 00:11:54,2.013040e+13,17,UDP,172.10.0.40,172.255.255.255,138,138,0,0,0,201,0,243,0,1,0,0
3,1.364948e+09,2013-04-03 00:11:57,2.013040e+13,6,TCP,10.156.215.83,172.10.0.7,42593,80,0,0,0,170,336,448,506,5,3,0
4,1.364948e+09,2013-04-03 00:12:00,2.013040e+13,6,TCP,10.170.32.110,172.20.0.4,42612,80,0,0,3,1870,79850,5730,84250,70,80,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21526133,1.365033e+09,2013-04-03 23:57:49,2.013040e+13,6,TCP,10.15.7.85,172.20.0.15,26886,80,0,0,11,19,503,297,619,5,2,0
21526134,1.365033e+09,2013-04-03 23:57:49,2.013040e+13,6,TCP,10.15.7.85,172.20.0.15,27614,80,0,0,5,19,503,297,619,5,2,0
21526135,1.365033e+09,2013-04-03 23:57:49,2.013040e+13,6,TCP,10.15.7.85,172.20.0.15,26887,80,0,0,11,19,503,297,619,5,2,0
21526136,1.365033e+09,2013-04-03 23:57:49,2.013040e+13,6,TCP,10.15.7.85,172.20.0.15,27978,80,0,0,2,19,503,297,619,5,2,0


In [6]:
%%time
# make a query
query = '''
        SELECT
            a.firstSeenSrcIp as source,
            a.firstSeenDestIp as destination,
            count(a.firstSeenDestPort) as targetPorts,
            SUM(a.firstSeenSrcTotalBytes) as bytesOut,
            SUM(a.firstSeenDestTotalBytes) as bytesIn,
            SUM(a.durationSeconds) as durationSeconds,
            MIN(parsedDate) as firstFlowDate,
            MAX(parsedDate) as lastFlowDate,
            COUNT(*) as attemptCount
        FROM
            netflow a
        GROUP BY
            a.firstSeenSrcIp,
            a.firstSeenDestIp
            '''

# query the table
gdf = bc_cudf.sql(query)

CPU times: user 1.17 s, sys: 165 ms, total: 1.33 s
Wall time: 833 ms


In [7]:
# how's it look?
gdf.to_csv("bsql_basic_query_gpu_output.csv", index=False)